# Día 7 · Unidad 7 — pandas y NumPy (IV): columnas, filtros, nulos y duplicados

**Objetivos de aprendizaje**
- Seleccionar, crear, modificar y eliminar columnas de un DataFrame.
- Gestionar valores nulos (`dropna`/`fillna`) con criterio, no de forma automática.
- Detectar y tratar registros duplicados (`duplicated`/`drop_duplicates`).
- Filtrar filas con condiciones lógicas, `isin`, `isnull`, `between` y `query`.
- **Traducir lógica de Excel (fórmulas condicionales, `SUMAR.SI`, `CONTAR.SI`) a pandas** — el bloque de IA de este día — y validar el resultado contra un cálculo de control.

**Duración estimada:** 90-100 minutos.

**Requisitos previos:** `03_teoria_dataframes_basico.ipynb` de este mismo día.

In [ ]:
import numpy as np
import pandas as pd

cartera = pd.read_csv("../data/raw/cartera_polizas.csv", parse_dates=["fecha_alta", "fecha_baja"])
cartera.head()

## 1. Columnas

### Seleccionar

Para seleccionar una columna: `df["nombre_columna"]` — devuelve una `Series`. Para seleccionar varias a la vez, una **lista** de nombres entre corchetes: `df[["col_a", "col_b"]]` — esto devuelve un `DataFrame`, no una Series, aunque sea de una sola columna.

In [ ]:
print(type(cartera["producto"]))          # Series
print(type(cartera[["producto"]]))         # DataFrame (una sola columna, pero sigue siendo tabla)

cartera[["poliza_id", "producto", "prima_anual"]].head()

También se puede acceder con notación de punto (`cartera.producto`), pero solo funciona si el nombre de columna es un identificador válido de Python (sin espacios ni caracteres especiales) y no coincide con un método existente del DataFrame — por eso, la forma general y más segura es siempre `df["columna"]`.

### Modificar

Las operaciones sobre una columna numérica son *element-wise* (vectorizadas, como ya viste con NumPy): se aplican a cada valor sin bucles.

In [ ]:
(cartera["prima_anual"] * 1.08).head()  # prima con el 8% de IPS, sin tocar el DataFrame original

Sobre columnas de texto, los métodos de string se acceden con el descriptor `.str` — llamar directamente a un método de `str` sobre la columna (sin `.str`) falla, porque la columna es una `Series`, no un único string.

In [ ]:
try:
    cartera["provincia"].upper()  # falla: Series no tiene .upper()
except AttributeError as e:
    print("Error esperado:", e)

cartera["provincia"].str.upper().head()  # correcto

### Crear columnas nuevas

Asignar a una columna que no existe la crea automáticamente:

In [ ]:
cartera["prima_con_ips"] = round(cartera["prima_anual"] * 1.08, 2)
cartera[["poliza_id", "prima_anual", "prima_con_ips"]].head()

También se pueden crear columnas a partir de operaciones entre columnas existentes, y con `.str.split()` para trocear texto (por ejemplo, separar `nombre_asegurado` en nombre y apellido):

In [ ]:
cartera["nombre_pila"] = cartera["nombre_asegurado"].str.split().str[0]
cartera["apellidos"] = cartera["nombre_asegurado"].str.split(n=1).str[1]
cartera[["nombre_asegurado", "nombre_pila", "apellidos"]].head()

### `.apply()` con varias columnas (`axis=1`)

Cuando la lógica depende de **más de una columna a la vez**, `.apply()` sobre una selección de columnas con `axis=1` pasa cada fila entera (como una `Series`) a la función.

In [ ]:
def clasificar_riesgo(fila: pd.Series) -> str:
    """Clasifica el riesgo combinando producto y franquicia.

    Args:
        fila: fila del DataFrame con, al menos, 'producto' y 'franquicia'.

    Returns:
        str: 'revisar' si es AUTO con franquicia baja, 'ok' en el resto.
    """
    if fila["producto"] == "AUTO" and fila["franquicia"] < 300:
        return "revisar"
    return "ok"


cartera["marca_revision"] = cartera[["producto", "franquicia"]].apply(clasificar_riesgo, axis=1)
cartera[["producto", "franquicia", "marca_revision"]].head()

> ⚠️ **Error típico**: usar `.apply(..., axis=1)` para cálculos que podrían resolverse con una operación vectorizada directa entre columnas (`df["a"] + df["b"]`). `.apply()` con `axis=1` internamente sigue iterando fila a fila por dentro (mucho más lento con datasets grandes) — resérvalo para lógica que de verdad no se puede vectorizar directamente (como el `if`/`elif` de arriba), no como primera opción por costumbre.

### Eliminar columnas

`.drop(columnas, axis=1)` elimina columnas; `.drop(indices, axis=0)` (o sin `axis`, filas por defecto) elimina filas. Por defecto no modifica el DataFrame original — hace falta `inplace=True` o reasignar.

In [ ]:
cartera = cartera.drop(columns=["nombre_pila", "apellidos", "marca_revision"])
cartera.columns.tolist()

**Para ti:** crea una columna `prima_mensual` a partir de `prima_anual` (divide entre 12 y redondea a 2 decimales), y otra columna `franquicia_alta` que sea `True` cuando `franquicia >= 300` (pista: la comparación ya devuelve directamente `True`/`False`, no necesitas `.apply()` para esto).

In [ ]:
# Tu código aquí


## 2. Gestión de nulos

Un valor ausente en pandas se representa como `NaN` (*Not a Number*). Ante un nulo hay tres posibilidades: **eliminarlo**, **sustituirlo** o **dejarlo como información válida** (a veces un nulo significa algo por sí mismo, no es un error). La decisión correcta depende del contexto de negocio, nunca es automática.

`cartera` no tiene nulos "de fábrica" salvo `fecha_baja` (solo se rellena si la póliza está anulada — aquí un nulo *es* información: significa "sigue vigente"). Vamos a comprobarlo.

In [ ]:
cartera.isnull().sum()

Efectivamente: `fecha_baja` tiene nulos exactamente en las pólizas no anuladas. **Este es justo el caso de "no eliminar ni rellenar": el nulo es el dato correcto.**

In [ ]:
assert (cartera["fecha_baja"].isnull() == ~cartera["anulada"]).all(),     "fecha_baja debería ser nulo exactamente cuando la póliza no está anulada"
print("Confirmado: fecha_baja nulo <=> póliza no anulada")

### `.dropna()`: eliminar

Elimina filas (`axis=0`, por defecto) o columnas (`axis=1`) con nulos. `how="any"` elimina si hay **al menos un** nulo; `how="all"` solo si **todos** los valores de esa fila/columna son nulos.

In [ ]:
# Aquí NO tiene sentido hacer dropna() sobre fecha_baja: borraríamos todas las pólizas vigentes.
# Un ejemplo donde sí tendría sentido: si quisiéramos trabajar solo con pólizas YA anuladas.
solo_anuladas_con_fecha = cartera.dropna(subset=["fecha_baja"])
print(f"Pólizas anuladas con fecha_baja informada: {len(solo_anuladas_con_fecha)}")
print(f"Total de pólizas anuladas (columna 'anulada'): {cartera['anulada'].sum()}")

### `.fillna()`: sustituir

Se usa cuando el nulo **sí** es un dato ausente real, no información en sí mismo. Reglas de partida habituales:

- Columna numérica: media, mediana, o un valor de negocio con sentido (nunca `0` "porque sí", si `0` significa algo distinto).
- Columna de texto: la moda, o una categoría explícita tipo `"desconocido"`.

Como ejemplo (aquí no aplica a `cartera`, lo veremos de verdad con `compras_seguros.csv` en los ejercicios), la sintaxis es:

```python
df["columna"] = df["columna"].fillna(valor)
```

> 📊 **En Excel esto sería...** como `=SI(ESBLANCO(A2), "desconocido", A2)` arrastrado por toda la columna: sustituye el hueco por un valor por defecto, celda a celda, en vez de dejarlo en blanco.

> ⚠️ **Error típico**: rellenar nulos con la media/moda **antes** de decidir si el nulo tenía significado propio. Con `fecha_baja`, rellenarla "porque hay huecos" sin pensar destruiría la información de qué pólizas siguen vigentes — el primer paso siempre es preguntarte **por qué** falta el dato, no solo cómo taparlo.

## 3. Duplicados

`.duplicated()` marca con `True` las filas duplicadas (según todas las columnas, o un subconjunto con `subset`); `.drop_duplicates()` las elimina. `.unique()`/`.nunique()` dan los valores únicos de una columna y cuántos hay.

In [ ]:
print("Productos únicos:", cartera["producto"].unique())
print("Número de provincias distintas:", cartera["provincia"].nunique())

`cartera_polizas.csv` no tiene duplicados a propósito (cada `poliza_id` es único), así que aquí `.duplicated()` da todo `False`:

In [ ]:
print("¿Hay poliza_id duplicados?", cartera["poliza_id"].duplicated().any())

En los ejercicios de este día trabajarás con `compras_seguros.csv`, que **sí tiene duplicados a propósito** (documentado en `data/generators/README.md`) para practicar exactamente este flujo: detectar cuántos hay, decidir con qué criterio (`subset`, `keep`), y solo entonces eliminarlos con `.drop_duplicates()`. Antes de borrar un duplicado en un dataset real, siempre conviene mirar **por qué** está duplicado — a veces es un error de carga, a veces son dos eventos legítimos con la misma pinta.

## 4. Filtros

### Condiciones lógicas

Ya usaste esto con Series. Sobre un DataFrame, el patrón es idéntico: una condición booleana entre corchetes selecciona filas.

In [ ]:
cartera[cartera["prima_anual"] > 800].head()

In [ ]:
# Combinar condiciones: & (y), | (o), cada condición entre paréntesis
cartera[
    (cartera["producto"] == "AUTO")
    & (cartera["prima_anual"] > 800)
].head()

### `.isin()`

Para comprobar pertenencia a una lista de valores, más legible que encadenar varios `|`.

In [ ]:
cartera[cartera["provincia"].isin(["Barcelona", "Madrid", "Valencia"])].head()

### `.isnull()`

Para filtrar por presencia/ausencia de nulo — aquí, pólizas vigentes (sin `fecha_baja`):

In [ ]:
cartera[cartera["fecha_baja"].isnull()].head()

### `.between()`

Filtra un rango numérico (inclusive por defecto en ambos extremos).

In [ ]:
cartera[cartera["edad_tomador"].between(30, 40)].head()

### `.query()`

Permite escribir el filtro como una condición en formato texto — muy útil cuando la condición se construye dinámicamente con variables.

In [ ]:
cartera.query("producto == 'AUTO' and prima_anual > 800").head()

In [ ]:
producto_objetivo = "HOGAR"
umbral = 500
cartera.query("producto == @producto_objetivo and prima_anual > @umbral").head()

El prefijo `@` dentro de `.query()` referencia una variable de Python (no una columna) — es la forma habitual de parametrizar un filtro sin tener que reconstruir el string de la condición a mano.

**Para ti:** usando cualquiera de los cuatro métodos de filtro, obtén las pólizas de `producto` `"SALUD"` o `"VIDA"`, con `edad_tomador` mayor de 55, que **no** están anuladas. Comprueba con un `assert` que el resultado no está vacío y que todas las filas cumplen `anulada == False`.

In [ ]:
# Tu código aquí


## 5. Con ayuda de la IA: traducir lógica de Excel a pandas

Antes de la migración a Python, mucho de este análisis se hacía con fórmulas de Excel: `SUMAR.SI`, `CONTAR.SI`, `SI` anidados, `BUSCARV`... Una de las tareas más frecuentes y más rentables al empezar con pandas es **traducir una fórmula que ya conocéis** a su equivalente, en vez de aprenderlo todo desde cero.

Imaginad que en Excel tenéis esta fórmula, sobre una tabla con las mismas columnas que `cartera` (`producto` en la columna C, `prima_anual` en la J):

```
=SUMAR.SI(C:C, "AUTO", J:J)
```

Suma `prima_anual` de todas las filas donde `producto` es `"AUTO"`. Queremos el equivalente en pandas.

### Prompt sugerido

In [ ]:
prompt_excel_a_pandas = """
Contexto: tengo una hoja de Excel con columnas producto y prima_anual (entre
otras), y esta formula: =SUMAR.SI(C:C, "AUTO", J:J), que suma prima_anual
de las filas donde producto es "AUTO". Ahora tengo los mismos datos en un
DataFrame de pandas llamado cartera, con columnas 'producto' y
'prima_anual'.

Objetivo: dame el equivalente en pandas de esa formula SUMAR.SI, y tambien
el equivalente de CONTAR.SI(C:C, "AUTO") (cuantas filas tienen
producto == "AUTO").

Restricciones: quiero la forma idiomatica de pandas (indexado booleano o
.loc), no un bucle for manual. No necesito que crees columnas nuevas, solo
los dos valores (la suma y el conteo).

Formato de salida: dos lineas de codigo, una por formula, cada una con un
comentario breve indicando que formula de Excel traduce.
"""
print(prompt_excel_a_pandas)

**Qué se espera obtener**
- El equivalente de `SUMAR.SI`: algo como `cartera.loc[cartera["producto"] == "AUTO", "prima_anual"].sum()`.
- El equivalente de `CONTAR.SI`: algo como `(cartera["producto"] == "AUTO").sum()` o `(cartera["producto"] == "AUTO").sum()` — contar `True` en una máscara booleana es sumar la máscara, porque `True` vale `1`.

**Qué debes verificar antes de darlo por bueno**
- Que el resultado de pandas coincide **numéricamente** con un cálculo de control hecho a mano sobre un subconjunto pequeño — no basta con que el código "se vea razonable" o "no dé error". Esto es justo lo que pide `docs/uso_ia_en_el_curso.md`: validar contra un caso conocido, sobre todo en análisis de datos, donde una transformación puede aplicarse a una columna equivocada o silenciar algo sin avisar.
- Que las condiciones usadas (`==`, tipos de dato) coinciden exactamente con lo que hacía la fórmula de Excel — un error típico es comparar texto con distinta capitalización (`"Auto"` vs `"AUTO"`) y que el filtro "funcione" pero devuelva menos filas de las que debería, en silencio.

In [ ]:
# Caso de control pequeño y conocido: las primeras 5 filas de la cartera, calculado A MANO
control = cartera.head(5)[["producto", "prima_anual"]]
control

In [ ]:
# Cálculo manual, fila a fila, mirando la tabla de arriba (esto es la "verdad" con la que comparamos)
suma_manual_auto = sum(
    fila["prima_anual"] for _, fila in control.iterrows() if fila["producto"] == "AUTO"
)
conteo_manual_auto = sum(1 for _, fila in control.iterrows() if fila["producto"] == "AUTO")
print(f"Cálculo manual (control): suma={suma_manual_auto}, conteo={conteo_manual_auto}")

In [ ]:
# Equivalentes en pandas de SUMAR.SI y CONTAR.SI, aplicados al mismo subconjunto de control
suma_pandas_auto = control.loc[control["producto"] == "AUTO", "prima_anual"].sum()
conteo_pandas_auto = (control["producto"] == "AUTO").sum()

print(f"Resultado de pandas (control): suma={suma_pandas_auto}, conteo={conteo_pandas_auto}")

assert suma_pandas_auto == suma_manual_auto, "La suma de pandas no coincide con el cálculo manual"
assert conteo_pandas_auto == conteo_manual_auto, "El conteo de pandas no coincide con el cálculo manual"
print("Validado: el equivalente de pandas coincide con el cálculo manual de control.")

Solo **después** de validar contra el caso de control pequeño (donde puedes comprobar el resultado a ojo) tiene sentido aplicar la misma lógica al DataFrame completo, con confianza de que hace lo que crees que hace:

In [ ]:
suma_total_auto = cartera.loc[cartera["producto"] == "AUTO", "prima_anual"].sum()
conteo_total_auto = (cartera["producto"] == "AUTO").sum()

print(f"Suma total de primas AUTO: {suma_total_auto:,.2f} €")
print(f"Número de pólizas AUTO: {conteo_total_auto}")

Esta es la mecánica que se pide para **cualquier** bloque de IA de traducción Excel→pandas del curso: no basta con que el prompt sea bueno o que el código se ejecute — el criterio de aceptación es que el resultado cuadre con un cálculo de control hecho por vosotros, a mano, sobre un caso pequeño donde podáis verificarlo a ojo.

## Resumen

- Seleccionar columnas: `df["col"]` (Series) o `df[["col_a", "col_b"]]` (DataFrame); crear columnas nuevas es tan simple como asignar a un nombre que no existe todavía.
- Los métodos de texto sobre una columna necesitan el descriptor `.str` (`df["col"].str.upper()`, no `df["col"].upper()`).
- `.apply(axis=1)` pasa cada fila completa a una función — resérvalo para lógica que combine varias columnas y no se pueda vectorizar directamente, porque por dentro sigue iterando fila a fila.
- Ante un nulo: pregúntate primero **por qué** falta, no cómo taparlo — a veces (como `fecha_baja`) el nulo **es** el dato correcto, no un hueco que rellenar.
- `.duplicated()`/`.drop_duplicates()`/`.unique()`/`.nunique()` gestionan registros repetidos — investiga siempre la causa antes de borrar duplicados de un dataset real.
- Cuatro formas de filtrar filas: condición booleana directa, `.isin()` (pertenencia a una lista), `.isnull()` (nulos) y `.between()` (rango); `.query()` es una quinta vía, útil para condiciones parametrizadas con `@variable`.
- Traducir una fórmula de Excel (`SUMAR.SI`, `CONTAR.SI`) a pandas con ayuda de un asistente de IA es rápido — pero el resultado solo se acepta si cuadra con un cálculo de control hecho a mano sobre un caso pequeño, nunca solo porque "no dé error".

**Siguiente:** `05_ejercicios.ipynb`, con `compras_seguros.csv` — el dataset con nulos y duplicados a propósito que verás mencionado varias veces en este notebook.